In [1]:
import pandas as pd
import plotly.express as px

In [2]:
data = pd.read_csv('../data/raw/non-market-housing.csv', sep=';')
data.rename(columns={'Clientele- Families': 'Clientele - Families'}, inplace=True)
data

,Index Number,Name,Address,Project Status,Occupancy Year,Operator,Clientele - Families,Clientele - Seniors,Clientele - Other,Design - Accessible 1BR,...,Design - Adaptable 3BR,Design - Adaptable 4BR,Design - Standard 1BR,Design - Standard 2BR,Design - Standard 3BR,Design - Standard 4BR,Design - Standard Studio,Design - Standard Room,URL,Geom
0,6,Helen's Court Co-op,2137 W 1st Ave,Completed,1984.0,Helen's Court Co-op Housing Association,35,0,9,2.0,...,0.0,0.0,7.0,22.0,13.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.1536261, 49.27104183], ""..."
1,7,Laura Jamieson Co-op,1349 E 2nd Ave,Completed,1987.0,Laura Jamieson Co-op Housing Association,42,0,5,1.0,...,0.0,0.0,4.0,23.0,18.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07614157, 49.26900321], ..."
2,8,Westerdale Co-op,1507 E 2nd Ave,Completed,1984.0,Westerdale Co-op Housing Association,10,0,9,4.0,...,0.0,0.0,5.0,6.0,2.0,0.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.07304966, 49.26896961], ..."
3,18,Vancouver Native (4th Ave),1560 E 4th Ave,Completed,1989.0,BC Indigenous Housing Society,20,10,1,1.0,...,0.0,0.0,10.0,11.0,4.0,5.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0718679, 49.2666326], ""t..."
4,20,Northern Way Co-op,675 E 5th Ave,Completed,1985.0,Northern Way Co-op Housing Association,44,0,16,3.0,...,0.0,0.0,13.0,24.0,19.0,1.0,0.0,0.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.09065341, 49.26639799], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,929,NaN,998 E 19th Ave,Approved,NaN,NaN,0,0,105,22.0,...,NaN,NaN,37.0,26.0,12.0,NaN,30.0,NaN,NaN,"{""coordinates"": [-123.08438908, 49.25336844], ..."
637,934,NaN,1710-1730 E Pender St,Approved,NaN,Lu'Ma Native Housing Society,71,0,120,NaN,...,NaN,NaN,120.0,38.0,28.0,5.0,NaN,NaN,NaN,"{""coordinates"": [-123.07002235, 49.28001135], ..."
638,948,Brennan's Place,545 E Cordova,Completed,2024.0,Lookout Housing and Health Society,0,0,20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.0923813, 49.28240033], ""..."
639,989,Murray Hotel,1119 Hornby,Completed,2017.0,Atira Women’s Resource Society,0,0,95,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95.0,https://app.vancouver.ca/NonMarketHousing_NET/...,"{""coordinates"": [-123.12750154, 49.27940628], ..."


## Project status

In [3]:
px.histogram(data, 'Project Status')

## Occupancy year

In [4]:
px.histogram(data, 'Occupancy Year', nbins=50)

## Operator

In [5]:
top_10_operators = data.Operator.value_counts().sort_values(ascending=False)[:10].index
px.histogram(
    data.query("Operator in @top_10_operators"), 
    'Operator', 
    category_orders={'Operator': top_10_operators}
)

## Clientele

In [6]:
data['Clientele'] = data.filter(like='Clientele').sum(axis=1)
clients = ['Families', 'Seniors', 'Other']
for client in clients:
    data[f'{client} (%)'] = data[f'Clientele - {client}'] / data.Clientele * 100

In [7]:
clientele = [i + ' (%)' for i in clients]
px.imshow(data[clientele].sort_values(clientele).reset_index(drop=True))

## Number of rooms

In [8]:
data['Total Units'] = data.filter(like='Design').sum(axis=1)
room_types = ['1BR', '2BR', '3BR', '4BR', 'Studio', 'Room']
for br in room_types:
    data[f'{br} (%)'] = data.filter(like=br).sum(axis=1) / data['Total Units'] * 100

In [9]:
rooms = [i + ' (%)' for i in room_types]
px.imshow(data[rooms].sort_values(rooms).reset_index(drop=True))

## Accessibility

In [10]:
access_types = ['Accessible', 'Adaptable', 'Standard']
for ac in access_types:
    data[f'{ac} (%)'] = data.filter(like=ac).sum(axis=1) / data['Total Units'] * 100

Filter to only those with at least some accessible/adaptable rooms.

In [11]:
access = [i + ' (%)' for i in access_types]
accessible_data = data.loc[data['Standard (%)'] != 100]
px.imshow(accessible_data[access].sort_values(access).reset_index(drop=True))